# Adult Census Income

Dataset-specific starting notebook for the DataFrameSampler paper experiments.

Claim-specific role: generic mixed-type benchmark, useful for early distributional similarity, downstream utility, and configuration-effort checks.


## Setup

Run the downloader before executing this notebook:

```bash
python experiments/download_datasets.py
```

Dataset-specific choices live in `experiments/datasets.py`; the reusable execution path lives in `experiments/workflow.py`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "experiments").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.datasets import DATASET_CONFIGS
from experiments.exploration import column_distribution_summary, plot_column_distributions, plot_pairwise_features
from experiments.numeric_projection import plot_numeric_projection_triptych
from experiments.workflow import dataset_profile, experiment_paths, load_dataset, notebook_environment, run_configured_dataset_experiment, working_dataframe

DATASET_NAME = "adult"
CONFIG = DATASET_CONFIGS[DATASET_NAME]
PATHS = experiment_paths(CONFIG, root=ROOT)
notebook_environment(PATHS)


## Load Prepared Data

Exploration uses the prepared source dataframe before the experiment workflow fits or samples anything.


In [ ]:
dataframe = load_dataset(CONFIG, root=ROOT)
work = working_dataframe(dataframe, CONFIG)
profile = dataset_profile(dataframe)
dataframe.shape, dataframe.head()


## Dataset Profile

The reusable Adult configuration uses a fixed 2,500-row working sample so exact KNN starter experiments run quickly while preserving a reproducible path to the full benchmark.


In [ ]:
profile


## Exploratory Data Analysis

These cells run before `run_configured_dataset_experiment`. They summarize every column, plot per-column distributions, and show pairwise numeric feature relationships for a bounded working sample.


In [ ]:
column_distribution_summary(dataframe)


In [ ]:
plot_column_distributions(dataframe, title="Adult Census Income")


In [ ]:
plot_pairwise_features(work, target_column=CONFIG.target_column, title="Adult Census Income")


## Run Experiment

The full sampler/baseline workflow runs after the exploratory cells.


In [ ]:
result = run_configured_dataset_experiment(CONFIG, root=ROOT)
result.paths


## Starter DataFrameSampler Run

The workflow writes the generated starter sample, quick similarity report, and runtime row to `experiments/results/`.


In [ ]:
result.starter_run.fit_seconds, result.starter_run.sample_seconds, result.starter_run.generated.head()


In [ ]:
result.starter_run.similarity_report


## Numeric Projection Of Generated Data

In [ ]:
projection_path = ROOT / "experiments" / "figures" / f"{CONFIG.dataset_name}_numeric_projection.pdf"
_ = plot_numeric_projection_triptych(
    result.starter_run.sampler,
    result.working_dataframe,
    result.starter_run.generated,
    title=CONFIG.title,
    reducer="umap",
    random_state=CONFIG.random_state,
    output_path=projection_path,
)
projection_path

## Baseline and Configuration Comparison

The same workflow runs DataFrameSampler default/manual/LLM-style configurations and the simple baselines, then writes the comparison CSV.


In [ ]:
result.comparison
